## Import CSV's and Python Libraries

Run `pip install -r requirements.txt` in your terminal to install the packages necessary to run this notebook on your local machine

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
#Import models here

df = pd.read_csv("csv's/tm_stuff_plus.csv")
df.head()

,pitcher_name,pitcher_level,pitcher_throws,date,pitch_type,pitch_number,release_speed,spin_rate,induced_vertical_break,horizontal_break,...,horizontal_approach_angle,vertical_break,spin_efficiency,spin_axis,tilt,extension,release_height,release_side,plate_location_height,plate_location_side
0,Wyatt Gibson,College,Left,2023-11-24,FB,7,78.56706,1797.9653,11.55309,-11.18099,...,0.239720,-34.85415,0.728916,136.02661,10:30:00,4.78048,5.07959,-1.27562,1.02636,-0.02276
1,Wyatt Gibson,College,Left,2023-11-24,FB,8,78.57823,1930.6069,13.41417,-5.73724,...,-0.279315,-33.47556,0.738446,156.90613,11:15:00,4.44279,5.11428,-1.20311,2.62001,-0.93968
2,Wyatt Gibson,College,Left,2023-11-24,FB,9,76.36920,1792.9038,17.66483,-3.36008,...,-0.507281,-32.57337,0.623952,169.24277,11:45:00,4.34863,5.14368,-1.50641,5.08989,-1.67569
3,Wyatt Gibson,College,Left,2023-11-24,SI,10,77.15758,1813.2701,14.13264,-13.68481,...,-1.059715,-36.14442,0.600773,136.01645,10:30:00,3.85140,5.07517,-1.04206,2.05757,-0.78176
4,Wyatt Gibson,College,Left,2023-11-24,SI,11,77.35618,1718.2062,NaN,NaN,...,NaN,NaN,NaN,97.03653,09:15:00,4.64714,4.95282,-1.26937,NaN,NaN


## Clean Data

This cell cleans the data accounting for missing rows and standardizing metrics by inverting all left-handed pitcher metrics to be right-handed for consistency.

In [ ]:
# All columns that will be multiplied by -1 to make LHP metrics line up with RHP
Flip_Cols = [
    "horizontal_break",
    "spin_axis",
    "release_side",
            ]

REQUIRED_COLS = ['pitch_type',
                 'pitcher_throws',
                 'release_speed',
                 'spin_rate',
                 'induced_vertical_break',
                 'horizontal_break'
                ]

# Drop warmup pitches
if 'is_warmup' in df.columns:
    df = df[df['is_warmup'] != 1]

# Flip the columns
df.loc[df['pitcher_throws'] == 'Left', Flip_Cols] *= -1
# Flip spin axis
df.loc[df['pitcher_throws'] == 'Left', 'spin_axis'] = (360 - df.loc[df['pitcher_throws'] == 'Left', 'spin_axis']) % 360

# Drop rows missing essential pitch data
df = df.dropna(subset=REQUIRED_COLS).copy()


## Stuff+ Model

Each pitch in this stuff+ model has specific weights representing which metrics matter most per pitch type. These weights were determined by cross-referencing with the mlb statcast data from 2024-2026 to find which metrics led to the biggest change in run value per pitch. The features that had the biggest impact on that target variable were assigned heavier weights. These values were normalized to sum to 1 so that they could be treated as percentages. For instance, if velocity has a weight of .5 for fastballs, that would mean that velocity is responsible for 50% of the output of your stuff+ number for that specific pitch. This would make sense intuitively as velocity is a big part of having an elite fastball.

In [ ]:
print(df.columns)

Index(['pitcher_name', 'pitcher_team', 'pitcher_level', 'pitcher_throws',
       'date', 'pitch_type', 'pitch_number', 'release_speed', 'spin_rate',
       'induced_vertical_break', 'horizontal_break', 'vertical_approach_angle',
       'horizontal_approach_angle', 'vertical_break', 'spin_efficiency',
       'spin_axis', 'tilt', 'extension', 'release_height', 'release_side',
       'plate_location_height', 'plate_location_side'],
      dtype='str')


## Output Data